# Redes neuronales con gráfos (GNN)

## 1. Primer ejemplo de una GNN

In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid

In [8]:
# Cargamos el Dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')

In [16]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

In [17]:
model = GCN(dataset.num_features, 16, dataset.num_classes)

In [18]:
# Entrenamos al modelos
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
data = dataset[0]

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(1001):
    loss = train()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss:.4f}')

Epoch 0, Loss: 1.9379
Epoch 10, Loss: 1.7802
Epoch 20, Loss: 1.5922
Epoch 30, Loss: 1.4017
Epoch 40, Loss: 1.2216
Epoch 50, Loss: 1.0553
Epoch 60, Loss: 0.9042
Epoch 70, Loss: 0.7689
Epoch 80, Loss: 0.6502
Epoch 90, Loss: 0.5485
Epoch 100, Loss: 0.4631
Epoch 110, Loss: 0.3923
Epoch 120, Loss: 0.3342
Epoch 130, Loss: 0.2867
Epoch 140, Loss: 0.2479
Epoch 150, Loss: 0.2160
Epoch 160, Loss: 0.1897
Epoch 170, Loss: 0.1680
Epoch 180, Loss: 0.1498
Epoch 190, Loss: 0.1345
Epoch 200, Loss: 0.1217
Epoch 210, Loss: 0.1107
Epoch 220, Loss: 0.1013
Epoch 230, Loss: 0.0932
Epoch 240, Loss: 0.0862
Epoch 250, Loss: 0.0801
Epoch 260, Loss: 0.0747
Epoch 270, Loss: 0.0700
Epoch 280, Loss: 0.0658
Epoch 290, Loss: 0.0621
Epoch 300, Loss: 0.0588
Epoch 310, Loss: 0.0558
Epoch 320, Loss: 0.0532
Epoch 330, Loss: 0.0507
Epoch 340, Loss: 0.0485
Epoch 350, Loss: 0.0465
Epoch 360, Loss: 0.0447
Epoch 370, Loss: 0.0430
Epoch 380, Loss: 0.0415
Epoch 390, Loss: 0.0400
Epoch 400, Loss: 0.0387
Epoch 410, Loss: 0.0375
Epo

In [20]:
# Evaluación
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    print(f'Accuracy: {acc:.4f}')

test()

Accuracy: 0.8070


## 2. Teoría

### 2.1 Background

A graph $G$ is defined as a pair $G=(V,E)$, where:

* $V$ is the set of nodes or vertices $\{v_1, v_2, ..., v_n\}$
* $E$ is the set of edges $(e_i, e_j)$ that represents the relationships between nodes.
* The matrix $A$ is an $n \times n$ matrix where $A_{ij} = 1$ if there is a connection between nodes $i,\ j$ and 0 otherwise.
* Each node can have a feature vector $x_i\in \mathbb{R}^d$

Examples of data we can structure in a graph:

* Relations between people
* Atoms in a molecule
* Pixels relations for images structure

<b> What types of problems with graph structure data can we solve?</b>

First of all we can handle 3 diffetent level
<ol>
    <li> <b> Graph-level task.</b> Here we can predict a single property for a whole graph.
    </li>
    <li> <b> Node-level task.</b> We can predict some property for each node in a graph. 
    </li>
    <li> <b> Edge-level task.</b> We can predict a propety or the presence of edges in a graph.
    </li>
</ol>

### 2.2 Definition of GNN

When we work with the relation between the nodes using the adjacency matrix $A$ the complexity becomes $O(n^2_{nodes})$ and we can find differents representation of the same conexions, it means $A$ is not invariant under permutations. To improve this problem we use an adjancecy list, where the element $e_k$ represents the conections between the nodes $v_i,\ v_j$ and it is invariant under permutations.

<b>Definition</b>
A GNN is an optimizable transformation on all attributes of the graph that preserves graph simmetries (permutations invariant).

### Fuentes:
1. [A Gentle Introduction to Graph Neural Networks](https://distill.pub/2021/gnn-intro/)

In [61]:
import numpy as np

def P_atm(masas, dia):
    N = len(masas)
    C = (4 * 9.81)/(N * np.pi * (dia**2))
    sum = np.sum(masas)
    presion_Pa = C * sum
    Presion_atm = presion_Pa/101325
    return Presion_atm

peso_clips = 0.0188
masas_1 = np.array([823.5, 818., 865.5]) * 0.001 + peso_clips
masas_2 = np.array([677.5, 641., 726.]) * 0.001 + peso_clips
masas_3 = np.array([796.5, 783., 808.]) * 0.001 + peso_clips

diametro = 1.08 * 0.01

#masas_t = np.ndarray([masas_1, masas_2, masas_3])

masas_t = np.zeros([4,1])

masas_t[0] = P_atm(masas_1, diametro)
masas_t[1] = P_atm(masas_2, diametro)
masas_t[2] = P_atm(masas_3, diametro)
masas_t[3] = np.mean([masas_t[i] for i in range(3)])

for i in range(3):
    print(f"La presión del conjunto {i+1} es:", round(np.mean(masas_t[i]),3), "atm")

print("\nLa presión media es de:", round(masas_t[3,0], 6), "atm" )

La presión del conjunto 1 es: 0.903 atm
La presión del conjunto 2 es: 0.74 atm
La presión del conjunto 3 es: 0.861 atm

La presión media es de: 0.834703 atm
